# Credit decision explainer Random Forest

---
### 1. Imports and config

In [11]:
import re
import time
import warnings
import subprocess

import numpy as np
import pandas as pd
import requests
import joblib
import shap

warnings.filterwarnings("ignore")
OLLAMA_HOST = "http://localhost:11434"
MODEL_NAME = "phi4-mini-reasoning:latest"
MODEL_PATH = "random_forest_model.pkl" 

---
### 2. Start Ollama

In [12]:
def start_ollama(wait_sec=30):
    try:
        requests.get(OLLAMA_HOST, timeout=2)
        print("Ollama is already running")
        return
    except requests.RequestException:
        pass
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(wait_sec):
        time.sleep(1)
        try:
            requests.get(OLLAMA_HOST, timeout=2)
            print("Ollama started")
            return
        except requests.RequestException:
            pass
    raise RuntimeError("Ollama did not start - try running 'ollama serve' in a terminal.")
start_ollama()

Ollama is already running


---
### 3. Load the Random Forest

In [13]:
model = joblib.load(MODEL_PATH)
FEATURES = list(model.feature_names_in_)
explainer = shap.TreeExplainer(model)
print(f"Loaded {type(model).__name__}: {model.n_estimators} trees, max_depth {model.max_depth}, {len(FEATURES)} features")

Loaded RandomForestClassifier: 100 trees, max_depth 5, 14 features


---
### 4. Knowledge for the prompt

In [14]:
VAR_INFO = {'newTradelines24Months': {'desc': 'New tradelines opened in the past 24 months', 'rel': 1, 'iv': 0.199, 'min': 0, 'p1': 0, 'median': 2, 'p90': 8, 'p99': 28, 'max': 224},
    'active_pl': {'desc': 'Count of active personal loans', 'rel': 1, 'iv': 0.163, 'min': 0, 'p1': 0, 'median': 0, 'p90': 2, 'p99': 6, 'max': 29},
    'count_small_loans': {'desc': 'Number of small-ticket loans', 'rel': 1, 'iv': 0.129, 'min': 0, 'p1': 0, 'median': 0, 'p90': 2, 'p99': 17, 'max': 292},
    'overdue_amount': {'desc': 'Total overdue amount (Rs)', 'rel': 1, 'iv': 0.136, 'min': 0, 'p1': 0, 'median': 0, 'p90': 1180, 'p99': 297400.0, 'max': 434761588},
    'income': {'desc': "Applicant's monthly income (Rs)", 'rel': -1, 'iv': 0.197, 'min': 0.11, 'p1': 10000, 'median': 50482, 'p90': 150000, 'p99': 500000.0, 'max': 100000000},
    'max_dpd_6_months': {'desc': 'Maximum days past due (DPD) in the last 6 months', 'rel': 1, 'iv': 0.129, 'min': 0, 'p1': 0, 'median': 0, 'p90': 0, 'p99': 880.2, 'max': 999},
    'enquiries': {'desc': 'Total number of credit enquiries', 'rel': 1, 'iv': 0.103, 'min': 0, 'p1': 0, 'median': 6, 'p90': 33, 'p99': 95, 'max': 462},
    'dpd_thirty': {'desc': '30-day days-past-due (DPD) bucket count', 'rel': 1, 'iv': 0.17, 'min': 0, 'p1': 0, 'median': 0, 'p90': 2, 'p99': 7, 'max': 65},
    'cc_util': {'desc': 'Credit card utilization percentage', 'rel': 1, 'iv': 0.092, 'min': -4986.0, 'p1': -1.209, 'median': 0.9211, 'p90': 64.43, 'p99': 101.8, 'max': 12750.0},
    'credit_history': {'desc': 'Length of credit history (months)', 'rel': -1, 'iv': 0.07, 'min': 0, 'p1': 2, 'median': 71, 'p90': 192, 'p99': 281, 'max': 24276},
    'recent_overdue': {'desc': 'Recent overdue flag (1 = yes, 0 = no)', 'rel': 1, 'iv': 0.141, 'min': 0, 'p1': 0, 'median': 0, 'p90': 1, 'p99': 1, 'max': 1},
    'past_overdue': {'desc': 'Past overdue flag (1 = yes, 0 = no)', 'rel': 1, 'iv': 0.143, 'min': 0, 'p1': 0, 'median': 0, 'p90': 1, 'p99': 1, 'max': 1},
    'months_since_recent_tradeline': {'desc': 'Months since most recent tradeline activity', 'rel': -1, 'iv': 0.135, 'min': 0, 'p1': 0, 'median': 7, 'p90': 37, 'p99': 98, 'max': 246},
    'lti': {'desc': 'Loan-to-income ratio', 'rel': 1, 'iv': 0.09, 'min': 0.0003108, 'p1': 0.1791, 'median': 1.8, 'p90': 5.2, 'p99': 10.61, 'max': 578200.0},}

BUCKETS = {"b05": (0.038753, 7.8, 6.9),
    "b04": (0.031813, 3.7, 4.1),
    "b03": (0.027560, 2.4, 2.0),
    "b02": (0.021400, 1.4, 1.4),
    "b01": (0.0,      0.7, 0.9),}
PORTFOLIO_DR_TRAIN, PORTFOLIO_DR_TEST = 3.2, 2.6    # %

---
### 5. Score the applicant (probability, bucket, SHAP)

In [15]:
def score(applicant):
    X = pd.DataFrame([applicant])[FEATURES]
    pd_ = float(model.predict_proba(X)[0, 1])

    bucket = str(np.where(pd_ >= 0.038753, "b05",
                 np.where(pd_ >= 0.031813, "b04",
                 np.where(pd_ >= 0.027560, "b03",
                 np.where(pd_ >= 0.021400, "b02", "b01")))))

    sv = explainer.shap_values(X)
    sv1 = sv[1][0] if isinstance(sv, list) else (sv[0, :, 1] if np.ndim(sv) == 3 else sv[0])
    base = explainer.expected_value
    base = float(base[1]) if np.ndim(base) else float(base)
    assert abs(base + sv1.sum() - pd_) < 1e-3, "SHAP values do not add up to the model score"
    return pd_, bucket, base, dict(zip(FEATURES, map(float, sv1)))


def position(var, value):
    i = VAR_INFO[var]
    if value > i["max"] or value < i["min"]: return "OUTSIDE the training range"
    if value > i["p99"]:return "very high (above the 99th percentile of training data)"
    if value > i["p90"]: return "high (above the 90th percentile)"
    if value > i["median"]: return "above the median"
    if value < i["p1"]: return "very low (below the 1st percentile)"
    return "at or below the median (typical)"

---
### 6. Prompt and LLM call

In [16]:
def build_prompt(applicant, pd_, bucket, base, contrib):
    _, tr, te = BUCKETS[bucket]
    rows = []
    for k in sorted(contrib, key=lambda f: -abs(contrib[f])):
        i = VAR_INFO[k]
        direction = "higher value -> higher default risk" if i["rel"] > 0 else "higher value -> lower default risk"
        rows.append(f"- {k} = {applicant[k]:,}  | contribution {contrib[k] * 100:+.2f} percentage points "
            f"({'raises' if contrib[k] > 0 else 'lowers' if contrib[k] < 0 else 'no effect on'} risk)\n"
            f"    meaning: {i['desc']}\n"
            f"    relationship with default: {direction} (monotonic in the model); information value {i['iv']}\n"
            f"    training data: typical range {i['p1']} to {i['p99']}, median {i['median']}, extreme max {i['max']:,}\n"
            f"    this applicant: {position(k, applicant[k])}")
    return f"""You are a credit risk assistant. A Random Forest model has ALREADY scored this personal-loan applicant.
Explain in plain language, for a credit underwriter, why the score is what it is. Do not recalculate the score.
Use only the facts below; do not invent numbers. No approve/reject policy is provided, so do not recommend a decision.

MODEL SCORE (computed by code):
- Predicted probability of default (90+ days past due within 12 months of booking): {pd_ * 100:.2f}%
- Starting point (average applicant in training): {base * 100:.2f}%. The contributions below add up to the difference.
- Risk bucket: {bucket} on a scale from b01 (lowest risk) to b05 (highest risk). Buckets are quintiles of the training score.
- Historical default rate of bucket {bucket}: {tr}% in training, {te}% in test, versus a portfolio average of {PORTFOLIO_DR_TRAIN}% (training) and {PORTFOLIO_DR_TEST}% (test).

VARIABLES, SORTED BY SIZE OF THEIR EFFECT ON THIS SCORE (SHAP contributions, in percentage points of default probability; positive raises risk, negative lowers it). Contributions are measured against the average applicant, so a value can be above the median and still lower the score slightly:
{chr(10).join(rows)}

Write about 250 words in this format:
**Summary:** one sentence with the predicted default probability, the bucket and what that bucket has meant historically.
**Main factors raising risk:** the top 3 positive contributions - say the applicant's value, what it means, and why that raises default risk.
**Factors lowering risk:** the biggest negative contributions, if any.
**Data notes:** mention any value flagged as outside the training range or above the 99th percentile; otherwise omit this section."""


def ask_llm(prompt):
    r = requests.post(f"{OLLAMA_HOST}/api/chat", timeout=900, json={
        "model": MODEL_NAME, "stream": False,
        "messages": [{"role": "user", "content": prompt}]})
    r.raise_for_status()
    text = r.json()["message"]["content"]
    return re.sub(r"(?s)^.*</think>|<think>.*?</think>", "", text).strip()   # drop the model's thinking block

---
### 7. Inputs

In [17]:
# ---------------- fill in the applicant ----------------
APPLICANT = {
    "newTradelines24Months": 6,               # typical 0 to 28 (median 2); training min 0, max 224
    "active_pl": 2,                           # typical 0 to 6 (median 0); training min 0, max 29
    "count_small_loans": 3,                   # typical 0 to 17 (median 0); training min 0, max 292
    "overdue_amount": 12000,                  # typical 0 to 297400.0 (median 0); training min 0, max 434761588
    "income": 32000,                          # typical 10000 to 500000.0 (median 50482); training min 0.11, max 100000000
    "max_dpd_6_months": 15,                   # typical 0 to 880.2 (median 0); training min 0, max 999
    "enquiries": 12,                          # typical 0 to 95 (median 6); training min 0, max 462
    "dpd_thirty": 1,                          # typical 0 to 7 (median 0); training min 0, max 65
    "cc_util": 75,                            # typical -1.209 to 101.8 (median 0.9211); training min -4986.0, max 12750.0
    "credit_history": 40,                     # typical 2 to 281 (median 71); training min 0, max 24276
    "recent_overdue": 1,                      # typical 0 to 1 (median 0); training min 0, max 1
    "past_overdue": 1,                        # typical 0 to 1 (median 0); training min 0, max 1
    "months_since_recent_tradeline": 2,       # typical 0 to 98 (median 7); training min 0, max 246
    "lti": 2.5,                               # typical 0.1791 to 10.61 (median 1.8); training min 0.0003108, max 578200.0
}
# --------------------------------------------------------

---
### 8. Run

In [18]:
missing = [f for f in FEATURES if f not in APPLICANT or pd.isna(APPLICANT[f])]
if missing:
    raise ValueError(f"Missing values for: {missing}")
for f in ("recent_overdue", "past_overdue"):
    if APPLICANT[f] not in (0, 1):
        raise ValueError(f"{f} must be 0 or 1")

pd_, bucket, base, contrib = score(APPLICANT)
print(f"Predicted default probability: {pd_ * 100:.2f}%   |   risk bucket: {bucket} (b01 lowest ... b05 highest)\n")
print("Contribution to the score (percentage points):")
for k in sorted(contrib, key=lambda f: -abs(contrib[f])):
    print(f"  {k:32s} {APPLICANT[k]:>12,}   {contrib[k] * 100:+6.2f}   {position(k, APPLICANT[k])}")

print("\nAsking the LLM to explain (can take a minute) ...\n")
try:
    print(ask_llm(build_prompt(APPLICANT, pd_, bucket, base, contrib)))
except Exception as e:
    print("LLM call failed:", e)
    print("Check that Ollama is running and the model is installed (run 'ollama list' in a terminal).")

Predicted default probability: 8.31%   |   risk bucket: b05 (b01 lowest ... b05 highest)

Contribution to the score (percentage points):
  overdue_amount                         12,000    +1.11   high (above the 90th percentile)
  count_small_loans                           3    +0.80   high (above the 90th percentile)
  max_dpd_6_months                           15    +0.68   high (above the 90th percentile)
  recent_overdue                              1    +0.46   above the median
  income                                 32,000    +0.45   at or below the median (typical)
  past_overdue                                1    +0.38   above the median
  months_since_recent_tradeline               2    +0.29   at or below the median (typical)
  credit_history                             40    +0.29   at or below the median (typical)
  cc_util                                    75    +0.28   high (above the 90th percentile)
  newTradelines24Months                       6    +0.27   above th